## Historical Archive Notice

This notebook is preserved for chronology and debugging history only. It is not the official midterm-compliant reproduction or submission path. Use `scripts/train_raw_baseline.py` and `notebooks/kaggle_submit_raw_baseline.ipynb` instead.


# Lean Single-Pass Submission Notebook

This is the only supported submission notebook.

It keeps one deterministic generation path:
1. setup / Drive checkout
2. package check
3. imports + config
4. model load
5. SVG helpers
6. generation helpers
7. submission build
8. final review

The planning pass, revision pass, and intermediate pass CSVs are intentionally removed.


In [33]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT_ON_DRIVE = DRIVE_ROOT / "svg-kaggle-comptetition"
OUTPUT_ROOT = PROJECT_ROOT_ON_DRIVE / "submission_outputs"
CANONICAL_DATASET_NAME = "train_canonicalized_candidate.csv"
CANONICAL_DATASET_DIR = PROJECT_ROOT_ON_DRIVE / "datasets"
LEGACY_PROJECT_ROOT = DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition"
LEGACY_DATASET_DIR = LEGACY_PROJECT_ROOT / "datasets"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def looks_like_repo_root(path: Path) -> bool:
    return path.exists() and ((path / ".git").exists() or (path / "README.md").exists())


def copy_project_tree(source: Path, destination: Path):
    shutil.copytree(
        source,
        destination,
        ignore=shutil.ignore_patterns(
            ".git",
            "__pycache__",
            ".ipynb_checkpoints",
            "runs",
            "submission_outputs",
        ),
    )


if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)

checkout_source = ""
try:
    subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])
    checkout_source = f"git clone: {GIT_REPO_URL}"
except subprocess.CalledProcessError as clone_error:
    drive_repo_source = next(
        (candidate for candidate in [PROJECT_ROOT_ON_DRIVE, LEGACY_PROJECT_ROOT] if looks_like_repo_root(candidate)),
        None,
    )
    if drive_repo_source is None:
        raise RuntimeError(
            "Git clone failed and no Drive repo copy was found at the expected project roots."
        ) from clone_error
    copy_project_tree(drive_repo_source, CHECKOUT_PATH)
    checkout_source = f"Drive repo copy: {drive_repo_source}"
(CHECKOUT_PATH / "datasets").mkdir(parents=True, exist_ok=True)

for required_name in ["test.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    candidate_paths = [
        PROJECT_ROOT_ON_DRIVE / required_name,
        LEGACY_PROJECT_ROOT / required_name,
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            break
    else:
        raise FileNotFoundError(f"Could not find {required_name} in Google Drive.")

canonical_dataset_destination = CHECKOUT_PATH / "datasets" / CANONICAL_DATASET_NAME
canonical_dataset_candidates = [
    CANONICAL_DATASET_DIR / CANONICAL_DATASET_NAME,
    LEGACY_DATASET_DIR / CANONICAL_DATASET_NAME,
    PROJECT_ROOT_ON_DRIVE / CANONICAL_DATASET_NAME,
    LEGACY_PROJECT_ROOT / CANONICAL_DATASET_NAME,
    CHECKOUT_PATH / "datasets" / CANONICAL_DATASET_NAME,
    CHECKOUT_PATH / CANONICAL_DATASET_NAME,
]
for candidate in canonical_dataset_candidates:
    if candidate.exists():
        if candidate != canonical_dataset_destination:
            shutil.copy2(candidate, canonical_dataset_destination)
        break
else:
    raise FileNotFoundError(
        f"Could not find canonicalized dataset {CANONICAL_DATASET_NAME} in Google Drive datasets/."
    )

os.chdir(CHECKOUT_PATH)
print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Checkout source: {checkout_source}")
print(f"Persistent output root: {OUTPUT_ROOT}")
print(f"Canonical dataset: {canonical_dataset_destination}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Error: [('/content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_candidate.gsheet', '/content/svg-kaggle-comptetition/datasets/train_canonicalized_candidate.gsheet', "[Errno 95] Operation not supported: '/content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_candidate.gsheet'"), ('/content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_candidate (1).gsheet', '/content/svg-kaggle-comptetition/datasets/train_canonicalized_candidate (1).gsheet', "[Errno 95] Operation not supported: '/content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_candidate (1).gsheet'")]

## Package Check

This cell installs the runtime packages needed by the notebook without using shell magics.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "trl": "trl",
    "cairosvg": "cairosvg",
    "lxml": "lxml",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


## Imports And Config

This cell imports the runtime dependencies, keeps the existing artifact discovery behavior, and defines the single-pass output paths.


In [ ]:
import copy
import io
import re
import shutil
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from lxml import etree
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

ET.register_namespace("", "http://www.w3.org/2000/svg")

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SVG_NS = "http://www.w3.org/2000/svg"
ARTIFACT_PREFIX = "qwen25coder15b_canon_len2048"
ADAPTER_PATH = CHECKOUT_PATH / f"{ARTIFACT_PREFIX}_adapter"
MERGED_PATH = CHECKOUT_PATH / f"{ARTIFACT_PREFIX}_merged"
DRIVE_PROJECT_CANDIDATES = [
    PROJECT_ROOT_ON_DRIVE,
    DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition",
]
DRIVE_LEGACY_ADAPTER_PATHS = [root / "svg-lora-adapter" for root in DRIVE_PROJECT_CANDIDATES]
DRIVE_LEGACY_MERGED_PATHS = [root / "svg-model-merged" for root in DRIVE_PROJECT_CANDIDATES]
RUNS_ROOT_CANDIDATES = [root / "runs" for root in DRIVE_PROJECT_CANDIDATES]
MERGED_WEIGHT_FILES = (
    "model.safetensors",
    "model.safetensors.index.json",
    "pytorch_model.bin",
    "pytorch_model.bin.index.json",
)
ADAPTER_REQUIRED_FILES = (
    "adapter_config.json",
    "adapter_model.safetensors",
)


def has_model_weights(path: Path) -> bool:
    return path.exists() and any((path / name).exists() for name in MERGED_WEIGHT_FILES)


def has_adapter_files(path: Path) -> bool:
    return path.exists() and all((path / name).exists() for name in ADAPTER_REQUIRED_FILES)


def first_matching_path(paths, predicate):
    for candidate in paths:
        if predicate(candidate):
            return candidate
    return None


def latest_run_artifact(run_roots, artifact_dir_name, predicate):
    for runs_root in run_roots:
        if not runs_root.exists():
            continue
        for run_dir in sorted([path for path in runs_root.iterdir() if path.is_dir()], reverse=True):
            candidate = run_dir / artifact_dir_name
            if predicate(candidate):
                return candidate
    return None


TEST_CSV = CHECKOUT_PATH / "test.csv"
CANONICAL_DATASET_NAME = "train_canonicalized_candidate.csv"
CANONICAL_DATASET_CSV = CHECKOUT_PATH / "datasets" / CANONICAL_DATASET_NAME
SUBMISSION_CSV = OUTPUT_ROOT / "submission.csv"
DEBUG_CSV = OUTPUT_ROOT / "submission_debug.csv"

if not CANONICAL_DATASET_CSV.exists():
    canonical_dataset_candidates = [
        CHECKOUT_PATH / "datasets" / CANONICAL_DATASET_NAME,
        CHECKOUT_PATH / CANONICAL_DATASET_NAME,
        PROJECT_ROOT_ON_DRIVE / "datasets" / CANONICAL_DATASET_NAME,
        PROJECT_ROOT_ON_DRIVE / CANONICAL_DATASET_NAME,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / "datasets" / CANONICAL_DATASET_NAME,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / CANONICAL_DATASET_NAME,
    ]
    resolved_dataset = first_matching_path(canonical_dataset_candidates, lambda path: path.exists())
    if resolved_dataset is None:
        raise FileNotFoundError(
            f"Could not find {CANONICAL_DATASET_NAME} in the checkout or Drive datasets paths."
        )
    CANONICAL_DATASET_CSV.parent.mkdir(parents=True, exist_ok=True)
    if resolved_dataset != CANONICAL_DATASET_CSV:
        shutil.copy2(resolved_dataset, CANONICAL_DATASET_CSV)

SVG_MAX_NEW_TOKENS = 1536
SVG_BATCH_SIZE = 200
RUN_LIMIT = None  # Set to 10 for a quick smoke test in Colab.

MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
RENDER_SIZE = 256
STRICT_VIEWBOX = f"0 0 {RENDER_SIZE} {RENDER_SIZE}"
STRICT_BOX = (0.0, 0.0, float(RENDER_SIZE), float(RENDER_SIZE))
DIRECT_ROOT_TAGS = {"defs", "title", "desc", "style"}
DRAWABLE_TAGS = {"path", "rect", "circle", "ellipse", "line", "polyline", "polygon", "use", "text", "image"}
DEBUG_COLUMNS = [
    "id",
    "reason",
    "gate_reason",
    "source_gate_reason",
    "normalized_gate_reason",
    "normalization_status",
    "strict_contract",
    "strict_issues",
    "collapsed",
    "hit_token_cap",
    "generated_tokens",
    "failure_reasons",
    "extracted_svg",
    "final_svg",
]

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter",
}

FALLBACK_SVG = (
    f'<svg xmlns="{SVG_NS}" viewBox="{STRICT_VIEWBOX}" width="{RENDER_SIZE}" height="{RENDER_SIZE}">'
    f'<rect width="{RENDER_SIZE}" height="{RENDER_SIZE}" fill="white"/>'
    '</svg>'
)

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)

print(f"Test CSV: {TEST_CSV}")
print(f"Canonicalized dataset CSV: {CANONICAL_DATASET_CSV}")
print(f"Final submission: {SUBMISSION_CSV}")
print(f"Final debug: {DEBUG_CSV}")
print(f"Run limit: {RUN_LIMIT}")


## Load Model

This cell keeps the current merged-model-first loading path, with the same temporary base-plus-LoRA fallback if merged weights are not present.


In [ ]:
local_merged_ready = has_model_weights(MERGED_PATH)
local_adapter_ready = has_adapter_files(ADAPTER_PATH)
drive_merged_path = first_matching_path(DRIVE_LEGACY_MERGED_PATHS, has_model_weights)
if drive_merged_path is None:
    drive_merged_path = latest_run_artifact(RUNS_ROOT_CANDIDATES, f"{ARTIFACT_PREFIX}_merged", has_model_weights)
drive_adapter_path = first_matching_path(DRIVE_LEGACY_ADAPTER_PATHS, has_adapter_files)
if drive_adapter_path is None:
    drive_adapter_path = latest_run_artifact(RUNS_ROOT_CANDIDATES, f"{ARTIFACT_PREFIX}_adapter", has_adapter_files)

if not local_merged_ready and drive_merged_path is not None:
    if MERGED_PATH.exists():
        shutil.rmtree(MERGED_PATH)
    shutil.copytree(drive_merged_path, MERGED_PATH)
    local_merged_ready = True
    print(f"Copied merged model from Drive: {drive_merged_path}")
elif MERGED_PATH.exists() and not local_merged_ready:
    print(f"Merged model directory exists at {MERGED_PATH}, but no weight files were found.")

if not local_adapter_ready and drive_adapter_path is not None:
    if ADAPTER_PATH.exists():
        shutil.rmtree(ADAPTER_PATH)
    shutil.copytree(drive_adapter_path, ADAPTER_PATH)
    local_adapter_ready = True
    print(f"Copied adapter from Drive: {drive_adapter_path}")
elif ADAPTER_PATH.exists() and not local_adapter_ready:
    print(f"Adapter directory exists at {ADAPTER_PATH}, but required files were not found.")

if local_merged_ready:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MERGED_PATH,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    MODEL_SOURCE = f"merged model at {MERGED_PATH}"
else:
    if not local_adapter_ready:
        raise FileNotFoundError(
            "Merged model weights were not found, and the LoRA adapter is also unavailable. "
            f"Checked local merged path: {MERGED_PATH}. "
            f"Checked legacy Drive merged paths: {DRIVE_LEGACY_MERGED_PATHS}. "
            f"Checked Drive run roots: {RUNS_ROOT_CANDIDATES}. "
            f"Checked local adapter path: {ADAPTER_PATH}. "
            f"Checked legacy Drive adapter paths: {DRIVE_LEGACY_ADAPTER_PATHS}."
        )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH), local_files_only=True)
    MODEL_SOURCE = f"base model + LoRA adapter at {ADAPTER_PATH}"
    print("Merged model not found. Using the temporary base-plus-LoRA fallback.")
    print("Remove this fallback path once the selected merged weights are reliably present in Drive runs.")

model.eval()
if model.config.pad_token_id is None and tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id

for attr_name in ("temperature", "top_p", "top_k"):
    if hasattr(model.generation_config, attr_name):
        setattr(model.generation_config, attr_name, None)

print(f"Loaded model source: {MODEL_SOURCE}")
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("model pad:", model.config.pad_token_id)


## SVG Helpers

This cell keeps the safe extraction, repair, XML recovery, validation, and strict `256x256` canonicalization helpers.


In [ ]:
EVENT_HANDLER_RE = re.compile(r"\son[a-zA-Z]+\s*=", re.IGNORECASE)
EXTERNAL_HREF_RE = re.compile(r"\s(?:href|xlink:href)\s*=\s*[\"']\s*(?:https?:|//)", re.IGNORECASE)
REMOTE_REF_RE = re.compile(r"@import\b|url\(\s*[\"']?(?:https?:|//)", re.IGNORECASE)
ROOT_TAG_RE = re.compile(r"<svg\b[^>]*>", flags=re.IGNORECASE | re.DOTALL)


def strip_namespace(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag


def extract_svg(text: str) -> str:
    text = str(text or "").strip()

    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()

    if "SVG:" in text:
        text = text.split("SVG:", 1)[1].strip()

    start = text.find("<svg")
    if start != -1:
        return text[start:].strip()

    return text


def render_svg(svg: str, size: int = RENDER_SIZE):
    try:
        png = cairosvg.svg2png(
            bytestring=svg.encode("utf-8"),
            output_width=size,
            output_height=size,
        )
        img = Image.open(io.BytesIO(png)).convert("RGB")
        return np.array(img)
    except Exception:
        return None


def get_attr_value(opening_tag: str, attr_name: str):
    pattern = rf"(\s{re.escape(attr_name)}\s*=\s*)([\"'])(.*?)\2"
    match = re.search(pattern, opening_tag, flags=re.IGNORECASE | re.DOTALL)
    if match is None:
        return None
    return match.group(3)


def format_number(value: float) -> str:
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    text = f"{value:.12g}"
    if "e" not in text and "." in text:
        text = text.rstrip("0").rstrip(".")
    return text


def parse_numeric_attr(value):
    if value is None:
        return None
    match = re.fullmatch(r"\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*(px)?\s*", str(value))
    if match is None:
        return None
    numeric = float(match.group(1))
    if numeric <= 0:
        return None
    return numeric


def parse_viewbox(value):
    if value is None:
        return None
    pieces = [piece for piece in re.split(r"[\s,]+", str(value).strip()) if piece]
    if len(pieces) != 4:
        return None
    try:
        x, y, width, height = [float(piece) for piece in pieces]
    except ValueError:
        return None
    if width <= 0 or height <= 0:
        return None
    return (x, y, width, height)


def derive_source_box(root: ET.Element):
    viewbox = parse_viewbox(root.attrib.get("viewBox"))
    if viewbox is not None:
        return viewbox, "viewBox"

    width = parse_numeric_attr(root.attrib.get("width"))
    height = parse_numeric_attr(root.attrib.get("height"))
    if width is not None and height is not None:
        return (0.0, 0.0, width, height), "width_height"

    return STRICT_BOX, "default"


def clone_element(element: ET.Element) -> ET.Element:
    return copy.deepcopy(element)


def serialize_svg_element(root: ET.Element) -> str:
    return ET.tostring(root, encoding="unicode")


def rebuild_svg_root(root: ET.Element, source_box, source_kind: str) -> tuple[str, str]:
    new_root = ET.Element(f"{{{SVG_NS}}}svg")
    new_root.set("viewBox", STRICT_VIEWBOX)
    new_root.set("width", str(RENDER_SIZE))
    new_root.set("height", str(RENDER_SIZE))

    for attr_name, attr_value in root.attrib.items():
        if attr_name in {"viewBox", "width", "height"}:
            continue
        new_root.set(attr_name, attr_value)

    source_x, source_y, source_width, source_height = source_box
    needs_transform = source_box != STRICT_BOX
    transform_group = None

    if needs_transform:
        sx = RENDER_SIZE / source_width
        sy = RENDER_SIZE / source_height
        tx = -source_x * sx
        ty = -source_y * sy
        transform_group = ET.Element(f"{{{SVG_NS}}}g")
        transform_group.set(
            "transform",
            f"matrix({format_number(sx)} 0 0 {format_number(sy)} {format_number(tx)} {format_number(ty)})",
        )

    group_inserted = False
    for child in list(root):
        child_copy = clone_element(child)
        child_tag = strip_namespace(child.tag)
        if child_tag in DIRECT_ROOT_TAGS or transform_group is None:
            new_root.append(child_copy)
            continue
        if not group_inserted:
            new_root.append(transform_group)
            group_inserted = True
        transform_group.append(child_copy)

    status = "unchanged" if not needs_transform else f"scaled_from_{source_kind}"
    return serialize_svg_element(new_root), status


def canonicalize_to_strict_256(svg: str):
    try:
        root = ET.fromstring(svg)
    except Exception:
        return svg, "parse_failed"

    if strip_namespace(root.tag) != "svg":
        return svg, "root_not_svg"

    source_box, source_kind = derive_source_box(root)
    return rebuild_svg_root(root, source_box, source_kind)


def strict_contract_issues(svg: str) -> list[str]:
    issues: list[str] = []
    opening_match = ROOT_TAG_RE.search(svg)
    if opening_match is None:
        issues.append("strict_parse_failed")
        return issues

    opening_tag = opening_match.group(0)
    if get_attr_value(opening_tag, "xmlns") != SVG_NS:
        issues.append("missing_xmlns")

    try:
        root = ET.fromstring(svg)
    except Exception:
        issues.append("strict_parse_failed")
        return issues

    if strip_namespace(root.tag) != "svg":
        issues.append("root_not_svg")
        return issues

    if root.attrib.get("viewBox") != STRICT_VIEWBOX:
        issues.append("viewbox_not_exact")

    if root.attrib.get("width") != str(RENDER_SIZE) or root.attrib.get("height") != str(RENDER_SIZE):
        issues.append("width_height_not_exact")

    return issues


def validity_gate(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 0, "svg_too_long_or_empty"

    svg = svg.strip()
    if len(svg) > MAX_SVG_LENGTH:
        return 0, "svg_too_long_or_empty"
    if EVENT_HANDLER_RE.search(svg):
        return 0, "disallowed_attr:event_handler"
    if EXTERNAL_HREF_RE.search(svg):
        return 0, "disallowed_ref:external_href"
    if REMOTE_REF_RE.search(svg):
        return 0, "disallowed_ref:remote_url"

    try:
        root = ET.fromstring(svg)
    except Exception:
        return 0, "parse_failed"

    if strip_namespace(root.tag) != "svg":
        return 0, "root_not_svg"

    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)
        if tag not in ALLOWED_TAGS:
            return 0, f"disallowed_tag:{tag}"
        if tag == "path":
            path_count += 1

    if path_count > MAX_PATH_COUNT:
        return 0, "too_many_paths"
    if render_svg(svg) is None:
        return 0, "render_failed"
    return 1, "valid"


def repair_svg(svg: str) -> str:
    if not svg:
        return svg

    svg = svg.strip()
    start = svg.find("<svg")
    if start != -1:
        svg = svg[start:]
    if "SVG:" in svg:
        svg = svg.split("SVG:", 1)[-1].strip()
    if "</svg>" in svg:
        end = svg.rfind("</svg>")
        svg = svg[: end + len("</svg>")]
    if "<svg" in svg and "</svg>" not in svg:
        svg += "</svg>"
    svg = re.sub(r"[A-Za-z0-9.\-]+$", "", svg)
    svg = re.sub(r"<[^>]*$", "", svg)
    return svg


def recover_svg_with_lxml(svg: str) -> str:
    if not svg or "<svg" not in svg:
        return svg
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg.encode("utf-8"), parser=parser)
        if root is None:
            return svg
        return etree.tostring(root, encoding="unicode")
    except Exception:
        return svg


def looks_collapsed(svg: str) -> bool:
    try:
        root = ET.fromstring(svg)
    except Exception:
        return True

    drawables = [elem for elem in root.iter() if strip_namespace(elem.tag) in DRAWABLE_TAGS]
    if not drawables:
        return True

    path_drawables = [elem for elem in drawables if strip_namespace(elem.tag) == "path"]
    if path_drawables and all(not elem.attrib.get("d", "").strip() for elem in path_drawables):
        non_path_drawables = [elem for elem in drawables if strip_namespace(elem.tag) != "path"]
        if not non_path_drawables:
            return True

    total_elems = sum(1 for _ in root.iter())
    return total_elems <= 1


def summarize_stage_failure(source_gate_reason: str, normalized_gate_reason: str) -> str:
    parts = []
    if source_gate_reason and source_gate_reason != "valid":
        parts.append(f"source:{source_gate_reason}")
    if normalized_gate_reason and normalized_gate_reason != "valid" and normalized_gate_reason != source_gate_reason:
        parts.append(f"normalized:{normalized_gate_reason}")
    if parts:
        return ";".join(parts)
    return source_gate_reason or normalized_gate_reason or "unknown_failure"


def candidate_from_svg(raw_text: str, extracted_svg: str, stage_svg: str, reason: str):
    source_valid, source_gate_reason = validity_gate(stage_svg)
    if source_valid == 0:
        return None, summarize_stage_failure(source_gate_reason, "")

    final_svg, normalization_status = canonicalize_to_strict_256(stage_svg)
    normalized_valid, normalized_gate_reason = validity_gate(final_svg)
    if normalized_valid == 0:
        return None, summarize_stage_failure(source_gate_reason, normalized_gate_reason)

    strict_issues = strict_contract_issues(final_svg)
    collapsed = looks_collapsed(final_svg)
    if strict_issues or collapsed:
        failure_reason = "strict_contract_failed" if strict_issues else "collapsed"
        return None, failure_reason

    candidate = {
        "reason": reason,
        "gate_reason": normalized_gate_reason,
        "source_gate_reason": source_gate_reason,
        "normalized_gate_reason": normalized_gate_reason,
        "normalization_status": normalization_status,
        "strict_contract": True,
        "strict_issues": "",
        "collapsed": False,
        "hit_token_cap": False,
        "generated_tokens": 0,
        "failure_reasons": "",
        "extracted_svg": extracted_svg,
        "final_svg": final_svg,
    }
    return candidate, normalized_gate_reason


def clean_svg_output(raw_text: str):
    cleaned_raw_text = str(raw_text or "")
    extracted_svg = extract_svg(cleaned_raw_text)
    failures = []

    raw_candidate, raw_failure = candidate_from_svg(cleaned_raw_text, extracted_svg, extracted_svg, "valid")
    if raw_candidate is not None:
        return raw_candidate
    failures.append(f"raw={raw_failure}")

    repaired_svg = repair_svg(extracted_svg)
    repaired_candidate, repaired_failure = candidate_from_svg(cleaned_raw_text, extracted_svg, repaired_svg, "repaired")
    if repaired_candidate is not None:
        repaired_candidate["failure_reasons"] = "|".join(failures)
        return repaired_candidate
    failures.append(f"repaired={repaired_failure}")

    xml_svg = recover_svg_with_lxml(repaired_svg)
    xml_candidate, xml_failure = candidate_from_svg(cleaned_raw_text, extracted_svg, xml_svg, "xml")
    if xml_candidate is not None:
        xml_candidate["failure_reasons"] = "|".join(failures)
        return xml_candidate
    failures.append(f"xml={xml_failure}")

    fallback_issues = strict_contract_issues(FALLBACK_SVG)
    return {
        "reason": "fallback",
        "gate_reason": failures[-1].split("=", 1)[-1] if failures else "fallback",
        "source_gate_reason": "",
        "normalized_gate_reason": "",
        "normalization_status": "fallback",
        "strict_contract": not fallback_issues,
        "strict_issues": ",".join(fallback_issues),
        "collapsed": False,
        "hit_token_cap": False,
        "generated_tokens": 0,
        "failure_reasons": "|".join(failures),
        "extracted_svg": extracted_svg,
        "final_svg": FALLBACK_SVG,
    }


fallback_strict_issues = strict_contract_issues(FALLBACK_SVG)
if fallback_strict_issues:
    raise ValueError(f"Fallback SVG is not strict-contract compliant: {fallback_strict_issues}")


In [ ]:
strict_result = clean_svg_output(
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256" width="256" height="256"><circle cx="128" cy="128" r="64"/></svg>'
)
assert strict_result["reason"] == "valid"
assert strict_contract_issues(strict_result["final_svg"]) == []
assert "transform=" not in strict_result["final_svg"]

scaled_200 = clean_svg_output(
    '<svg viewBox="0 0 200 200" width="200" height="200"><rect x="20" y="20" width="160" height="160"/></svg>'
)
assert scaled_200["reason"] == "valid"
assert strict_contract_issues(scaled_200["final_svg"]) == []
assert 'matrix(1.28 0 0 1.28 0 0)' in scaled_200["final_svg"]

scaled_36 = clean_svg_output(
    '<svg width="36" height="36"><path d="M0 0 L36 36"/></svg>'
)
assert scaled_36["reason"] == "valid"
assert strict_contract_issues(scaled_36["final_svg"]) == []
assert f'matrix({format_number(256 / 36)} 0 0 {format_number(256 / 36)} 0 0)' in scaled_36["final_svg"]

missing_xmlns = clean_svg_output(
    '<svg viewBox="0 0 256 256" width="256" height="256"><rect width="256" height="256"/></svg>'
)
assert missing_xmlns["reason"] == "valid"
assert strict_contract_issues(missing_xmlns["final_svg"]) == []

repaired = clean_svg_output(
    '<svg width="200" height="200"><path d="M0 0 L200 200"/>'
)
assert repaired["reason"] == "repaired"
assert strict_contract_issues(repaired["final_svg"]) == []

xml_recovered = clean_svg_output(
    '<svg><g><path d="M0 0 L10 10"/></svg>'
)
assert xml_recovered["reason"] == "xml"
assert strict_contract_issues(xml_recovered["final_svg"]) == []

collapsed = clean_svg_output(
    '<svg viewBox="0 0 256 256" width="256" height="256"><path/></svg>'
)
assert collapsed["reason"] == "fallback"

print("SVG helper self-checks passed.")


## Generation Helpers

This cell defines the deterministic batch generation helper plus the single-pass submission builder.


In [ ]:
def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return value != 0
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def load_test_df(test_csv_path=TEST_CSV, limit=RUN_LIMIT):
    test_df = pd.read_csv(test_csv_path)
    test_df = test_df.dropna(subset=["id", "prompt"]).copy()
    test_df["prompt"] = test_df["prompt"].astype(str)
    if limit is not None:
        test_df = test_df.head(limit).copy()
    return test_df.reset_index(drop=True)


def save_dataframe(df: pd.DataFrame, path, label: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved {label} to: {path}")


def build_svg_input(prompt: str) -> str:
    return f"Prompt: {prompt}\nSVG:\n"


def generate_text_batch(inputs_text, batch_size, max_new_tokens, do_sample=False):
    batch_outputs = []
    next_index = 0
    current_batch_size = max(1, batch_size)
    progress_bar = tqdm(total=len(inputs_text))

    while next_index < len(inputs_text):
        batch_inputs_text = inputs_text[next_index:next_index + current_batch_size]
        inputs = None
        outputs = None
        try:
            inputs = tokenizer(
                batch_inputs_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=do_sample,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            input_width = int(inputs["input_ids"].shape[1])
            generated_only = outputs[:, input_width:].detach().cpu()
            decoded_batch = tokenizer.batch_decode(generated_only, skip_special_tokens=True)
            eos_token_id = tokenizer.eos_token_id
            for raw_text, generated_ids in zip(decoded_batch, generated_only):
                generated_token_ids = generated_ids.tolist()
                if eos_token_id is not None and eos_token_id in generated_token_ids:
                    eos_index = generated_token_ids.index(eos_token_id)
                    effective_token_ids = generated_token_ids[:eos_index]
                    hit_token_cap = False
                else:
                    effective_token_ids = generated_token_ids
                    hit_token_cap = len(effective_token_ids) >= max_new_tokens
                batch_outputs.append(
                    {
                        "raw_text": raw_text,
                        "generated_tokens": len(effective_token_ids),
                        "hit_token_cap": hit_token_cap,
                    }
                )
            next_index += len(batch_inputs_text)
            progress_bar.update(len(batch_inputs_text))
        except torch.cuda.OutOfMemoryError:
            if current_batch_size == 1:
                raise
            current_batch_size = max(1, current_batch_size // 2)
            print(f"CUDA OOM. Retrying with batch size {current_batch_size}.")
        finally:
            del inputs
            del outputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    progress_bar.close()
    return batch_outputs


def build_candidate(generation_output: dict):
    candidate = clean_svg_output(generation_output.get("raw_text", ""))
    candidate["generated_tokens"] = int(generation_output.get("generated_tokens", 0))
    candidate["hit_token_cap"] = as_bool(generation_output.get("hit_token_cap", False))
    return candidate


def build_submission(test_df, svg_batch_size=SVG_BATCH_SIZE):
    prompts = test_df["prompt"].astype(str).tolist()
    generation_outputs = generate_text_batch(
        [build_svg_input(prompt) for prompt in prompts],
        batch_size=svg_batch_size,
        max_new_tokens=SVG_MAX_NEW_TOKENS,
        do_sample=False,
    )
    candidates = [build_candidate(output) for output in generation_outputs]

    submission_df = pd.DataFrame(
        {
            "id": test_df["id"].tolist(),
            "svg": [candidate["final_svg"] for candidate in candidates],
        }
    )

    debug_rows = []
    for row, candidate in zip(test_df.itertuples(index=False), candidates):
        debug_row = {column: candidate.get(column, "") for column in DEBUG_COLUMNS if column != "id"}
        debug_row["id"] = row.id
        debug_rows.append(debug_row)
    debug_df = pd.DataFrame(debug_rows)
    debug_df = debug_df[DEBUG_COLUMNS]

    print("Final reason counts:")
    print(debug_df["reason"].value_counts())
    print(f"Final strict-contract pass rate: {float(debug_df['strict_contract'].map(as_bool).mean()):.4f}")
    print(f"Token-cap hits: {int(debug_df['hit_token_cap'].map(as_bool).sum())}")
    return submission_df, debug_df


## Build Submission

This cell runs the single deterministic generation pass, saves `submission.csv` and `submission_debug.csv`, and performs lightweight smoke checks on the saved outputs.


In [ ]:
test_df = load_test_df(test_csv_path=TEST_CSV, limit=RUN_LIMIT)
submission_df, debug_df = build_submission(test_df)

save_dataframe(submission_df, SUBMISSION_CSV, "final submission")
save_dataframe(debug_df, DEBUG_CSV, "final debug")

forbidden_prefixes = ("plan_", "baseline_candidate_", "pass2_", "revise_")
unexpected_debug_columns = [
    column
    for column in debug_df.columns
    if any(column.startswith(prefix) for prefix in forbidden_prefixes)
]
if unexpected_debug_columns:
    raise AssertionError(f"Unexpected legacy debug columns: {unexpected_debug_columns}")

strict_failures = [svg for svg in submission_df["svg"].tolist() if strict_contract_issues(svg)]
if strict_failures:
    raise AssertionError(f"Found {len(strict_failures)} non-strict SVG outputs after canonicalization.")

print(f"Submission rows: {len(submission_df)}")
print(f"Debug rows: {len(debug_df)}")
display(submission_df.head())
display(debug_df.head())


## Final Review

This cell reloads the saved outputs, enforces the strict competition SVG wrapper on `submission.csv`, rewrites the saved file, and previews the normalized result.


In [ ]:
def ensure_competition_svg_wrapper(svg: str) -> str:
    if pd.isna(svg):
        raw_svg = ""
    else:
        raw_svg = str(svg).strip()
    if not raw_svg:
        return FALLBACK_SVG
    if "<svg" not in raw_svg.lower():
        raw_svg = (
            f'<svg xmlns="{SVG_NS}" viewBox="{STRICT_VIEWBOX}" width="{RENDER_SIZE}" height="{RENDER_SIZE}">'
            f"{raw_svg}"
            "</svg>"
        )
    return clean_svg_output(raw_svg)["final_svg"]

saved_submission_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
saved_debug_df = pd.read_csv(DEBUG_CSV, keep_default_na=False)

normalized_svgs = [ensure_competition_svg_wrapper(svg) for svg in saved_submission_df["svg"].tolist()]
rewrapped_rows = sum(
    original != normalized
    for original, normalized in zip(saved_submission_df["svg"].tolist(), normalized_svgs)
)
saved_submission_df = saved_submission_df.copy()
saved_submission_df["svg"] = normalized_svgs

strict_failures = [svg for svg in saved_submission_df["svg"].tolist() if strict_contract_issues(svg)]
if strict_failures:
    raise AssertionError(f"Found {len(strict_failures)} non-strict SVG outputs after wrapper normalization.")

save_dataframe(saved_submission_df, SUBMISSION_CSV, "rewrapped submission")

print(f"Saved submission rows: {len(saved_submission_df)}")
print(f"Rows rewritten during wrapper normalization: {rewrapped_rows}")
print(f"Saved debug rows: {len(saved_debug_df)}")
print("Saved debug columns:")
print(saved_debug_df.columns.tolist())

if "strict_contract" in saved_debug_df.columns:
    strict_mask = saved_debug_df["strict_contract"].map(as_bool)
    print(f"Saved strict-contract pass rate: {float(strict_mask.mean()):.4f}")
    print(f"Saved rows with strict issues: {int((~strict_mask).sum())}")

print("Saved reason counts:")
print(saved_debug_df["reason"].value_counts())
display(saved_submission_df.head())
display(saved_debug_df.head())
